#**Preparation**

In [1]:
!apt-get update
!apt-get install -y portaudio19-dev

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,332 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,083 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [

In [2]:
import sys
!"{sys.executable}" -m pip install pydub pyaudio pysrt librosa ffmpeg-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 609.3 kB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyaudio: filename=pyaudio-0.2.14-cp312-cp312-linux_x86_64.whl size=68667 sha256=b059d72f5a352eb142a98f4ff040382930b3d9f8bcd52269305c4a27870d206a
  Stored in directory: /root/.cache/pip/wheels/68/c7/33/c6a6b210cb5819ec5c219928c794a447742a7d86d21c0b92e6
  Created wheel for pysrt: filename=pysrt-1.1.2-py3-none-any.whl size=13443 sha256=094a7707ae40a62a0df14f129c85ed724398f3bc74307141a920ee4ded8ec60f
  Stored in directory: /root/.cache/pip/wheels/6a/36/54/2aa8dc961885dfa7b0ebd45a57505f25039d79b4ea0fd9f29d
Successfully built pyaudio pysrt


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


#**Trim With SRT**

In [ ]:
from moviepy.editor import VideoFileClip
import pysrt
import os
import re

# ===== Konfigurasi =====
video_path = "/content/drive/MyDrive/TADATASET/Sewudino/Sewu_Dino_(2023).mp4"
subtitle_path = "/content/drive/MyDrive/TADATASET/Sewudino/Sewu.Dino.2023.-id.srt"
output_directory = "/content/drive/MyDrive/TADATASET/Sewudino/clips"

# Margin trimming (ms)
starttime_margin = 0
endtime_margin = 0

# Filter subtitle (opsional, None kalau mau semua disimpan)
filter_keyword = None

os.makedirs(output_directory, exist_ok=True)

print("🎬 Memuat video dan subtitle...")
video = VideoFileClip(video_path)
subtitles = pysrt.open(subtitle_path)

print(f"✅ Video durasi: {video.duration:.2f} detik")
print(f"✅ Total subtitle: {len(subtitles)} baris\n")

def make_safe_filename(text, index):
    """Buat nama file aman untuk penyimpanan."""
    cleaned_text = re.sub(r'[\\/*?:"<>|]', '', text)
    cleaned_text = cleaned_text.replace("\n", " ").strip()

    safe_text = re.sub(r'^\W+', '', cleaned_text)
    safe_text = re.sub(r'\s+', '_', safe_text)

    if not safe_text:
        safe_text = f"clip_{index}"

    return safe_text[:50], cleaned_text  # (nama_aman, teks_asli)


# ===== Proses Pemotongan =====
for i, subtitle in enumerate(subtitles):
    if filter_keyword and filter_keyword.lower() not in subtitle.text.lower():
        continue

    start_time = max(subtitle.start.ordinal / 1000 - (starttime_margin / 1000), 0)
    end_time = min(subtitle.end.ordinal / 1000 + (endtime_margin / 1000), video.duration)

    # Skip subtitle di luar durasi video
    if start_time >= video.duration:
        print(f"⏭️  Subtitle {i} dilewati (mulai {start_time:.2f}s > durasi video {video.duration:.2f}s)")
        continue
    if end_time <= start_time:
        print(f"⚠️  Subtitle {i} dilewati (end_time <= start_time)")
        continue

    safe_name, original_text = make_safe_filename(subtitle.text, i)
    clip_output_path = os.path.join(output_directory, f"{safe_name}.mp4")

    try:
        clip = video.subclip(start_time, end_time)
        clip.write_videofile(
            clip_output_path,
            codec="libx264",
            audio_codec="aac",
            verbose=False,
            logger=None
        )
        print(f"✅ Saved: {os.path.basename(clip_output_path)}  ({start_time:.2f}s - {end_time:.2f}s)")
        print(f"   Subtitle: {original_text}\n")

    except Exception as e:
        print(f"❌ Gagal menyimpan clip {i}: {e}\n")

video.close()
print("🏁 Proses selesai! Semua potongan tersimpan di:", output_directory)


🎬 Memuat video dan subtitle...
✅ Video durasi: 7282.72 detik
✅ Total subtitle: 985 baris

✅ Saved: Saudari....mp4  (24.92s - 26.08s)
   Subtitle: Saudari...

✅ Saved: Apa,_Jum_-_Ayo_pergi._Saya_tidak....mp4  (26.88s - 29.00s)
   Subtitle: - Apa, Jum - Ayo pergi. Saya tidak...

✅ Saved: Apa_Anda_tidak_ingin_memandikannya.mp4  (29.08s - 32.00s)
   Subtitle: Apa Anda tidak ingin memandikannya

✅ Saved: Jum,_kenapa_kamu_gadis_Kliwon_takut.mp4  (32.83s - 36.42s)
   Subtitle: Jum, kenapa kamu gadis Kliwon takut

✅ Saved: Anda_dilahirkan_pada_tanggal_a_Kliwon_Jumat_ya.mp4  (36.46s - 37.96s)
   Subtitle: Anda dilahirkan pada tanggal a Kliwon Jumat ya

✅ Saved: Dengarkan_aku..._-_Hentikan!.mp4  (38.00s - 40.54s)
   Subtitle: - Dengarkan aku... - Hentikan!

✅ Saved: Cukup!.mp4  (40.62s - 41.96s)
   Subtitle: Cukup!

✅ Saved: Saya_merasa_panas_di_sini..mp4  (42.04s - 43.42s)
   Subtitle: Saya merasa panas di sini.

✅ Saved: Saya_sedang_membersihkan_rumah_memakai_pakaian_yan.mp4  (43.46s - 45.79s)

#**TANPA SRT**

In [ ]:
print("⏳ Memulai instalasi....")
# Instalasi MoviePy dan Whisper (untuk Speech-to-Text)
!pip install -q moviepy
!pip install -q git+https://github.com/openai/whisper.git

⏳ Memulai instalasi....
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os
import re
import whisper
from moviepy.editor import VideoFileClip
from google.colab import drive
import warnings
import json

# Menonaktifkan peringatan untuk tampilan yang lebih bersih
warnings.filterwarnings("ignore")
print("✅ Instalasi selesai dan pustaka diimpor.")

print("\n--- KONFIGURASI JALUR FILE ---")

# Ubah path ini sesuai dengan lokasi file Anda di Google Drive
video_path = "/content/drive/MyDrive/TADATASET/Danur2/danur-2-maddah-2018.mp4"

# Direktori output juga bisa langsung disimpan di Google Drive
output_directory = "/content/drive/MyDrive/TADATASET/Danur2/clipsspeech2"

# --- Konfigurasi Pemotongan ---
MIN_DURATION = 1.0       # <--- BATAS MINIMAL DURASI KLIP (DETIK)
starttime_margin = 0.1   # detik
endtime_margin = 0.1     # detik

# Filter ucapan (opsional, None kalau mau semua segmen ucapan disimpan)
filter_keyword = None

# --- Konfigurasi Transkripsi (Whisper) ---
whisper_model_name = "base"

# Cek apakah file video ada
if not os.path.exists(video_path):
    raise FileNotFoundError(f"❌ ERROR: File tidak ditemukan di jalur Drive: {video_path}\nPastikan jalur sudah benar dan Drive sudah ter-mount.")

os.makedirs(output_directory, exist_ok=True)
print(f"✅ File video yang akan diproses: {video_path}")
print(f"✅ Klip hasil akan disimpan di Drive: {output_directory}")

def make_safe_filename(text, index):
    """Buat nama file aman untuk penyimpanan dari transkrip."""
    cleaned_text = re.sub(r'[\\/*?:"<>|]', '', text)
    cleaned_text = cleaned_text.replace("\n", " ").strip()
    safe_text = re.sub(r'^\W+', '', cleaned_text)
    safe_text = re.sub(r'\s+', '_', safe_text)
    if not safe_text:
        safe_text = f"clip_{index}"
    return safe_text[:50], cleaned_text

def combine_segments(segments, min_duration):
    """
    Menggabungkan segmen Whisper yang berdekatan hingga durasinya mencapai
    atau melebihi batas minimal.

    Segmen yang berdekatan (bisa dari orang berbeda) akan digabungkan
    untuk memenuhi batas minimal 3 detik, memastikan klip tetap berdasarkan ucapan.
    """
    if not segments:
        return []

    combined = []
    current_segment = segments[0]

    for next_segment in segments[1:]:
        # Hitung durasi gabungan saat ini
        duration = next_segment['end'] - current_segment['start']

        if duration < min_duration:
            # Gabungkan dengan segmen berikutnya
            current_segment['end'] = next_segment['end']
            current_segment['text'] += next_segment['text']
            current_segment['id'] = str(current_segment['id']) + "-" + str(next_segment['id']) # Catat ID yang digabungkan
        else:
            # Durasi sudah cukup, simpan segmen saat ini dan mulai segmen baru
            combined.append(current_segment)
            current_segment = next_segment

    # Tambahkan segmen terakhir (baik yang sisa atau yang baru)
    if current_segment not in combined:
        combined.append(current_segment)

    # Lakukan pengecekan akhir, jika segmen terakhir masih kurang dari MIN_DURATION,
    # gabungkan dengan segmen sebelumnya yang sudah ada. (Opsional, tergantung kebijakan)
    if len(combined) > 1 and (combined[-1]['end'] - combined[-1]['start']) < min_duration:
        last_combined = combined.pop()
        combined[-1]['end'] = last_combined['end']
        combined[-1]['text'] += last_combined['text']
        combined[-1]['id'] = str(combined[-1]['id']) + "-" + str(last_combined['id'])

    return combined

try:
    # MEMUAT VIDEO DAN MODEL WHISPER
    print("\n--- INISIALISASI ---")
    print("🎬 Memuat video...")
    video = VideoFileClip(video_path)
    print(f"✅ Video durasi: {video.duration:.2f} detik")

    print(f"🧠 Memuat model Whisper ({whisper_model_name})...")
    model = whisper.load_model(whisper_model_name)
    print("✅ Model siap.")

    # TRANSKRIPSI (SPEECH-TO-TEXT)
    print("\n--- TRANSKRIPSI ---")
    print("📝 Proses transkripsi (ini mungkin memakan waktu lama)...")
    result = model.transcribe(video_path)
    raw_segments = result['segments']

    print(f"✅ Transkripsi selesai. Total segmen ucapan awal: {len(raw_segments)}")

    # PENGGABUNGAN SEGMENT BERDASARKAN DURASI MINIMAL
    segments = combine_segments(raw_segments, MIN_DURATION)
    print(f"🔄 Total segmen ucapan setelah digabung: {len(segments)}\n")


    # PROSES PEMOTONGAN BERBASIS SEGMENT UCAPAN YANG DIGABUNG
    print("--- PROSES PEMOTONGAN ---")
    for i, segment in enumerate(segments):
        text = segment['text']

        start_time_raw = segment['start']
        end_time_raw = segment['end']
        current_duration = end_time_raw - start_time_raw

        # Filter keyword
        if filter_keyword and filter_keyword.lower() not in text.lower():
            continue

        # Pengecekan Durasi Akhir
        if current_duration < MIN_DURATION and i != len(segments) -1:
            # Seharusnya sudah digabungkan, ini sebagai guardrail
            print(f"⚠️ Segmen {i} dilewati karena durasi ({current_duration:.2f}s) < {MIN_DURATION}s.")
            continue


        # Terapkan margin trimming
        start_time = max(start_time_raw - starttime_margin, 0)
        end_time = min(end_time_raw + endtime_margin, video.duration)

        # Skip jika waktu tidak valid
        if start_time >= video.duration or end_time <= start_time:
             print(f"⚠️ Segmen {i} dilewati (Waktu tidak valid).")
             continue

        # Buat nama file yang aman
        safe_name, original_text = make_safe_filename(text, i)
        clip_output_path = os.path.join(output_directory, f"{i:03d}_{safe_name}.mp4")

        try:
            # Memotong klip
            clip = video.subclip(start_time, end_time)

            # Menyimpan klip
            clip.write_videofile(
                clip_output_path,
                codec="libx264",
                audio_codec="aac",
                verbose=False,
                logger=None,
                temp_audiofile=f"/tmp/audio_{i}.m4a"
            )
            print(f"✅ Saved: {os.path.basename(clip_output_path)} (Durasi: {end_time - start_time:.2f}s)")
            print(f"      Ucapan: {original_text.strip()}\n")

        except Exception as e:
            print(f"❌ Gagal menyimpan clip {i}: {e}\n")

    video.close()

    print("\n🏁 Proses selesai! Semua potongan tersimpan di folder Drive Anda:", output_directory)

except FileNotFoundError as fnfe:
    print(fnfe)
except Exception as e:
    print(f"\n❌ TERJADI KESALAHAN KRITIS: {e}")

✅ Instalasi selesai dan pustaka diimpor.

--- KONFIGURASI JALUR FILE ---
✅ File video yang akan diproses: /content/drive/MyDrive/TADATASET/Danur2/danur-2-maddah-2018.mp4
✅ Klip hasil akan disimpan di Drive: /content/drive/MyDrive/TADATASET/Danur2/clipsspeech2

--- INISIALISASI ---
🎬 Memuat video...
✅ Video durasi: 5148.04 detik
🧠 Memuat model Whisper (base)...
✅ Model siap.

--- TRANSKRIPSI ---
📝 Proses transkripsi (ini mungkin memakan waktu lama)...
✅ Transkripsi selesai. Total segmen ucapan awal: 611
🔄 Total segmen ucapan setelah digabung: 610

--- PROSES PEMOTONGAN ---
✅ Saved: 000_Ayuh.mp4 (Durasi: 3.26s)
      Ucapan: Ayuh

✅ Saved: 001_Ayuh.mp4 (Durasi: 3.40s)
      Ucapan: Ayuh

✅ Saved: 002_N_proces.mp4 (Durasi: 2.44s)
      Ucapan: N proces

✅ Saved: 003_ye_laisser.mp4 (Durasi: 1.58s)
      Ucapan: ye laisser

✅ Saved: 004_ya.mp4 (Durasi: 1.54s)
      Ucapan: ya

✅ Saved: 005_cari.mp4 (Durasi: 2.44s)
      Ucapan: cari

✅ Saved: 006_que.mp4 (Durasi: 1.82s)
      Ucapan: que

⚠

In [ ]:
#>5Second akan diremove
import os
from moviepy.editor import VideoFileClip
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

print("--- 🎬 START: PROGRAM PEMBERSIHAN KLIP ---")

TARGET_DIR_PATH = "/content/drive/MyDrive/TADATASET/Danur2/clipsspeech2"
MAX_DURATION_SECONDS = 5.0
FILE_EXTENSION = ".mp4"

def cleanup_long_videos(directory: str, max_duration: float, extension: str):
    """
    Mencari dan menghapus file video yang durasinya melebihi batas yang ditentukan.
    """
    target_path = Path(directory)
    if not target_path.exists():
        print(f"❌ ERROR: Direktori output tidak ditemukan: {directory}")
        return

    print(f"\n** Mencari file {extension} dengan durasi > {max_duration:.1f} detik di:\n{target_path} **")

    files_deleted = 0
    total_files_checked = 0

    # Iterasi melalui semua file di direktori
    for item in target_path.glob(f"*{extension}"):

        # Hanya proses file
        if item.is_file():
            total_files_checked += 1
            filename = item.name

            try:
                # 1. Mendapatkan durasi video menggunakan moviepy
                clip = VideoFileClip(str(item))
                duration = clip.duration
                clip.close() # Penting untuk menutup klip setelah mendapatkan durasi

                print(f"  - File: {filename} | Durasi: {duration:.2f} detik")

                # 2. Cek apakah durasi melebihi batas
                if duration > max_duration:
                    print(f"    >>> Durasi {duration:.2f} detik > {max_duration:.1f} detik. MENGHAPUS FILE...")

                    # 3. Menghapus file
                    os.remove(item)
                    files_deleted += 1


            except Exception as e:
                # Menangani error jika file tidak dapat diproses (misalnya corrupt)
                print(f"    !!! GAGAL memproses {filename} (mungkin rusak): {e}")


    print("\n--- RINGKASAN PEMBERSIHAN ---")
    print(f"Total file {FILE_EXTENSION} yang dicek: {total_files_checked}")
    print(f"Total file yang dihapus: {files_deleted}")
    print("🏁 Pembersihan klip selesai.")


# Panggil fungsi utama dengan path yang dideklarasikan ulang
cleanup_long_videos(TARGET_DIR_PATH, MAX_DURATION_SECONDS, FILE_EXTENSION)

--- 🎬 START: PROGRAM PEMBERSIHAN KLIP ---

** Mencari file .mp4 dengan durasi > 5.0 detik di:
/content/drive/MyDrive/TADATASET/Danur2/clipsspeech2 **
  - File: 000_Ayuh.mp4 | Durasi: 3.28 detik
  - File: 001_Ayuh.mp4 | Durasi: 3.40 detik
  - File: 002_N_proces.mp4 | Durasi: 2.44 detik
  - File: 003_ye_laisser.mp4 | Durasi: 1.60 detik
  - File: 004_ya.mp4 | Durasi: 1.56 detik
  - File: 005_cari.mp4 | Durasi: 2.48 detik
  - File: 006_que.mp4 | Durasi: 1.84 detik
  - File: 008_computer.mp4 | Durasi: 1.40 detik
  - File: 009_berapa_berusia_lama_sekarang.mp4 | Durasi: 2.24 detik
  - File: 010_63_tahun.mp4 | Durasi: 2.24 detik
  - File: 011_gimana_mempertahunan_rumpang_kemudian.mp4 | Durasi: 2.24 detik
  - File: 012_di_pavilion_belakang_rumah.mp4 | Durasi: 2.24 detik
  - File: 013_apa_yang_pokok_ingat_pertama_dia_sekarang.mp4 | Durasi: 2.24 detik
  - File: 014_wajah_yang_cantik.mp4 | Durasi: 2.24 detik
  - File: 015_rambutnya_yang_berblombang_panjang_dan_wangi.mp4 | Durasi: 4.24 detik
  - Fi

#up to 5 minute

In [5]:
from moviepy.editor import VideoFileClip
import pysrt
import os
import re

# ===== Konfigurasi =====
video_path = "/content/drive/MyDrive/TADATASET/DearNathan/dear-nathan-2017.mp4"
subtitle_path = "/content/drive/MyDrive/TADATASET/DearNathan/Dear.Nathan.2017.NF.WEB-DL.srt"
output_directory = "/content/drive/MyDrive/TADATASET/DearNathan/clips5menit"

# Margin trimming (ms) - sekarang diterapkan di awal & akhir tiap KLIP (bukan per baris subtitle lagi)
starttime_margin = 0
endtime_margin = 0

# Filter subtitle (opsional, None kalau mau semua disimpan)
filter_keyword = None

# ===== Target durasi tiap klip =====
MIN_CLIP_DURATION = 3 * 60   # 180 detik (3 menit)
MAX_CLIP_DURATION = 5 * 60   # 300 detik (5 menit)

os.makedirs(output_directory, exist_ok=True)

print("🎬 Memuat video dan subtitle...")
video = VideoFileClip(video_path)
subtitles = pysrt.open(subtitle_path)

print(f"✅ Video durasi: {video.duration:.2f} detik")
print(f"✅ Total subtitle: {len(subtitles)} baris\n")

if filter_keyword:
    subtitles = [s for s in subtitles if filter_keyword.lower() in s.text.lower()]
    print(f"🔎 Setelah filter '{filter_keyword}': {len(subtitles)} baris\n")


def format_timestamp(seconds):
    total = int(seconds)
    h, rem = divmod(total, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}h{m:02d}m{s:02d}s" if h else f"{m:02d}m{s:02d}s"


def group_subtitles(subs, min_duration, max_duration):
    """Gabungkan baris-baris subtitle berurutan jadi satu grup sampai
    total durasinya minimal `min_duration` detik. `max_duration` adalah
    batas aman: kalau ada jeda dialog yang panjang di tengah, grup boleh
    sedikit melebihi batas ini daripada menghasilkan klip yang kependekan."""
    groups = []
    current_group = []

    for sub in subs:
        if current_group:
            group_start = current_group[0].start.ordinal / 1000
            current_duration = current_group[-1].end.ordinal / 1000 - group_start
            projected_duration = sub.end.ordinal / 1000 - group_start

            # Grup sekarang sudah cukup (>= minimum) tapi menambah subtitle
            # berikutnya bakal melebihi maksimum -> tutup dulu grup ini
            if current_duration >= min_duration and projected_duration > max_duration:
                groups.append(current_group)
                current_group = []

        current_group.append(sub)

        group_start = current_group[0].start.ordinal / 1000
        group_end = current_group[-1].end.ordinal / 1000
        if group_end - group_start >= min_duration:
            groups.append(current_group)
            current_group = []

    if current_group:
        if groups:
            groups[-1].extend(current_group)  # gabung sisa ke klip terakhir
        else:
            groups.append(current_group)

    return groups


groups = group_subtitles(subtitles, MIN_CLIP_DURATION, MAX_CLIP_DURATION)

print(f"📦 Total klip yang akan dibuat: {len(groups)}")
for i, group in enumerate(groups):
    g_start = group[0].start.ordinal / 1000
    g_end = group[-1].end.ordinal / 1000
    print(f"   Klip {i+1:02d}: {format_timestamp(g_start)} - {format_timestamp(g_end)} "
          f"({g_end - g_start:.1f} detik, {len(group)} subtitle)")
print()

# ===== Proses Pemotongan =====
for i, group in enumerate(groups):
    start_time = max(group[0].start.ordinal / 1000 - (starttime_margin / 1000), 0)
    end_time = min(group[-1].end.ordinal / 1000 + (endtime_margin / 1000), video.duration)

    if start_time >= video.duration:
        print(f"⏭️  Klip {i+1} dilewati (mulai {start_time:.2f}s > durasi video {video.duration:.2f}s)")
        continue
    if end_time <= start_time:
        print(f"⚠️  Klip {i+1} dilewati (end_time <= start_time)")
        continue

    clip_filename = f"clip_{i+1:03d}_{format_timestamp(start_time)}-{format_timestamp(end_time)}.mp4"
    clip_output_path = os.path.join(output_directory, clip_filename)

    try:
        clip = video.subclip(start_time, end_time)
        clip.write_videofile(
            clip_output_path,
            codec="libx264",
            audio_codec="aac",
            verbose=False,
            logger=None
        )
        print(f"✅ Saved: {clip_filename}  ({start_time:.2f}s - {end_time:.2f}s, {end_time-start_time:.1f} detik)\n")

    except Exception as e:
        print(f"❌ Gagal menyimpan klip {i+1}: {e}\n")

video.close()
print("🏁 Proses selesai! Semua potongan tersimpan di:", output_directory)

🎬 Memuat video dan subtitle...
✅ Video durasi: 5928.22 detik
✅ Total subtitle: 1448 baris

📦 Total klip yang akan dibuat: 30
   Klip 01: 00m01s - 03m02s (181.5 detik, 44 subtitle)
   Klip 02: 03m03s - 06m05s (181.7 detik, 50 subtitle)
   Klip 03: 06m05s - 09m06s (180.7 detik, 73 subtitle)
   Klip 04: 09m06s - 12m11s (184.8 detik, 57 subtitle)
   Klip 05: 12m11s - 15m12s (180.7 detik, 48 subtitle)
   Klip 06: 15m12s - 18m14s (181.9 detik, 63 subtitle)
   Klip 07: 18m21s - 21m23s (181.8 detik, 52 subtitle)
   Klip 08: 21m24s - 24m34s (190.2 detik, 53 subtitle)
   Klip 09: 24m34s - 27m35s (181.0 detik, 44 subtitle)
   Klip 10: 27m35s - 30m41s (186.0 detik, 56 subtitle)
   Klip 11: 30m50s - 33m51s (180.7 detik, 42 subtitle)
   Klip 12: 33m52s - 36m52s (180.8 detik, 65 subtitle)
   Klip 13: 36m52s - 39m57s (184.2 detik, 55 subtitle)
   Klip 14: 39m57s - 43m00s (182.4 detik, 40 subtitle)
   Klip 15: 43m00s - 46m05s (184.6 detik, 44 subtitle)
   Klip 16: 46m24s - 49m25s (181.2 detik, 42 subti

In [6]:
from moviepy.editor import VideoFileClip
import pysrt
import os
import re

# ===== Konfigurasi =====
video_path = "/content/drive/MyDrive/TADATASET/AbadiJaya/ABADINANJAYA-IDMX21-MV01.mp4"
subtitle_path = "/content/drive/MyDrive/TADATASET/AbadiJaya/The.Elixir.2025.1080p.WEB.h264-EDITH-id.srt"
output_directory = "/content/drive/MyDrive/TADATASET/AbadiJaya/clips5menit"

# Margin trimming (ms) - sekarang diterapkan di awal & akhir tiap KLIP (bukan per baris subtitle lagi)
starttime_margin = 0
endtime_margin = 0

# Filter subtitle (opsional, None kalau mau semua disimpan)
filter_keyword = None

# ===== Target durasi tiap klip =====
MIN_CLIP_DURATION = 3 * 60   # 180 detik (3 menit)
MAX_CLIP_DURATION = 5 * 60   # 300 detik (5 menit)

os.makedirs(output_directory, exist_ok=True)

print("🎬 Memuat video dan subtitle...")
video = VideoFileClip(video_path)
subtitles = pysrt.open(subtitle_path)

print(f"✅ Video durasi: {video.duration:.2f} detik")
print(f"✅ Total subtitle: {len(subtitles)} baris\n")

if filter_keyword:
    subtitles = [s for s in subtitles if filter_keyword.lower() in s.text.lower()]
    print(f"🔎 Setelah filter '{filter_keyword}': {len(subtitles)} baris\n")


def format_timestamp(seconds):
    total = int(seconds)
    h, rem = divmod(total, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}h{m:02d}m{s:02d}s" if h else f"{m:02d}m{s:02d}s"


def group_subtitles(subs, min_duration, max_duration):
    """Gabungkan baris-baris subtitle berurutan jadi satu grup sampai
    total durasinya minimal `min_duration` detik. `max_duration` adalah
    batas aman: kalau ada jeda dialog yang panjang di tengah, grup boleh
    sedikit melebihi batas ini daripada menghasilkan klip yang kependekan."""
    groups = []
    current_group = []

    for sub in subs:
        if current_group:
            group_start = current_group[0].start.ordinal / 1000
            current_duration = current_group[-1].end.ordinal / 1000 - group_start
            projected_duration = sub.end.ordinal / 1000 - group_start

            # Grup sekarang sudah cukup (>= minimum) tapi menambah subtitle
            # berikutnya bakal melebihi maksimum -> tutup dulu grup ini
            if current_duration >= min_duration and projected_duration > max_duration:
                groups.append(current_group)
                current_group = []

        current_group.append(sub)

        group_start = current_group[0].start.ordinal / 1000
        group_end = current_group[-1].end.ordinal / 1000
        if group_end - group_start >= min_duration:
            groups.append(current_group)
            current_group = []

    if current_group:
        if groups:
            groups[-1].extend(current_group)  # gabung sisa ke klip terakhir
        else:
            groups.append(current_group)

    return groups


groups = group_subtitles(subtitles, MIN_CLIP_DURATION, MAX_CLIP_DURATION)

print(f"📦 Total klip yang akan dibuat: {len(groups)}")
for i, group in enumerate(groups):
    g_start = group[0].start.ordinal / 1000
    g_end = group[-1].end.ordinal / 1000
    print(f"   Klip {i+1:02d}: {format_timestamp(g_start)} - {format_timestamp(g_end)} "
          f"({g_end - g_start:.1f} detik, {len(group)} subtitle)")
print()

# ===== Proses Pemotongan =====
for i, group in enumerate(groups):
    start_time = max(group[0].start.ordinal / 1000 - (starttime_margin / 1000), 0)
    end_time = min(group[-1].end.ordinal / 1000 + (endtime_margin / 1000), video.duration)

    if start_time >= video.duration:
        print(f"⏭️  Klip {i+1} dilewati (mulai {start_time:.2f}s > durasi video {video.duration:.2f}s)")
        continue
    if end_time <= start_time:
        print(f"⚠️  Klip {i+1} dilewati (end_time <= start_time)")
        continue

    clip_filename = f"clip_{i+1:03d}_{format_timestamp(start_time)}-{format_timestamp(end_time)}.mp4"
    clip_output_path = os.path.join(output_directory, clip_filename)

    try:
        clip = video.subclip(start_time, end_time)
        clip.write_videofile(
            clip_output_path,
            codec="libx264",
            audio_codec="aac",
            verbose=False,
            logger=None
        )
        print(f"✅ Saved: {clip_filename}  ({start_time:.2f}s - {end_time:.2f}s, {end_time-start_time:.1f} detik)\n")

    except Exception as e:
        print(f"❌ Gagal menyimpan klip {i+1}: {e}\n")

video.close()
print("🏁 Proses selesai! Semua potongan tersimpan di:", output_directory)

🎬 Memuat video dan subtitle...
✅ Video durasi: 7061.78 detik
✅ Total subtitle: 1383 baris

📦 Total klip yang akan dibuat: 34
   Klip 01: 00m27s - 03m28s (180.4 detik, 42 subtitle)
   Klip 02: 03m28s - 06m29s (181.3 detik, 41 subtitle)
   Klip 03: 06m50s - 10m13s (203.1 detik, 45 subtitle)
   Klip 04: 10m19s - 13m21s (181.9 detik, 53 subtitle)
   Klip 05: 13m21s - 16m22s (180.4 detik, 56 subtitle)
   Klip 06: 16m22s - 19m23s (180.3 detik, 49 subtitle)
   Klip 07: 19m23s - 22m24s (180.4 detik, 58 subtitle)
   Klip 08: 22m24s - 25m28s (183.7 detik, 58 subtitle)
   Klip 09: 25m29s - 28m29s (180.3 detik, 62 subtitle)
   Klip 10: 28m39s - 31m44s (184.7 detik, 27 subtitle)
   Klip 11: 31m48s - 35m09s (200.9 detik, 35 subtitle)
   Klip 12: 35m10s - 38m29s (199.3 detik, 44 subtitle)
   Klip 13: 38m30s - 41m30s (180.5 detik, 20 subtitle)
   Klip 14: 41m31s - 44m55s (204.4 detik, 50 subtitle)
   Klip 15: 45m03s - 48m04s (180.2 detik, 36 subtitle)
   Klip 16: 48m04s - 51m08s (183.6 detik, 40 subti

#**GANTI NAMA**

In [ ]:
import os
from openpyxl import Workbook

# Define the folder containing the files
folder_path = r"/content/drive/MyDrive/TADATASET/Sewudino/Normalisasi"

# Ensure the folder exists
if not os.path.exists(folder_path):
    print("Folder does not exist!")
    exit()

# Get a list of files and their last modified times
files = [(f, os.path.getmtime(os.path.join(folder_path, f)))
         for f in os.listdir(folder_path)
         if os.path.isfile(os.path.join(folder_path, f))]

# Sort files by last modified time (oldest first)
files.sort(key=lambda x: x[1])

# Rename files with the new format
for index, (file_name, _) in enumerate(files, start=1):
    # Get the file extension
    file_extension = os.path.splitext(file_name)[1]

    # Create the new file name
    new_file_name = f"01-SWDN-UT-{index:04d}{file_extension}"

    # Define full file paths
    old_path = os.path.join(folder_path, file_name)
    new_path = os.path.join(folder_path, new_file_name)

    # Rename the file
    os.rename(old_path, new_path)
    print(f"Renamed: {file_name} → {new_file_name}")

Renamed: 999_Hari.wav → 01-SWDN-UT-0001.wav
Renamed: 998_Hari.wav → 01-SWDN-UT-0002.wav
Renamed: ada_di_ruangan_ini..wav → 01-SWDN-UT-0003.wav
Renamed: Adakah_hal_aneh_yang_terjadi_saat_aku_pergi.wav → 01-SWDN-UT-0004.wav
Renamed: adalah_tuanku..wav → 01-SWDN-UT-0005.wav
Renamed: Ada_sesuatu_yang_perlu_saya_lakukan..wav → 01-SWDN-UT-0006.wav
Renamed: Abaikan_saja..wav → 01-SWDN-UT-0007.wav
Renamed: Ada_apa,_Sri_-_Sri!.wav → 01-SWDN-UT-0008.wav
Renamed: Ada_apa_pada_hari_itu.wav → 01-SWDN-UT-0009.wav
Renamed: Ada_tiga_di_antaranya..wav → 01-SWDN-UT-0010.wav
Renamed: 1.000_Hari.wav → 01-SWDN-UT-0011.wav
Renamed: Apa_yang_terjadi,_Sri.wav → 01-SWDN-UT-0012.wav
Renamed: Adik_perempuanku_juga_suka_tidur..wav → 01-SWDN-UT-0013.wav
Renamed: Aku_akan_memberitahumu_tentang_hal_itu..wav → 01-SWDN-UT-0014.wav
Renamed: Aku_bukan_siapa-siapa!.wav → 01-SWDN-UT-0015.wav
Renamed: Aku_akan_pergi..wav → 01-SWDN-UT-0016.wav
Renamed: Aku_harus_melakukan_ini,_Sri..wav → 01-SWDN-UT-0017.wav
Renamed: Aku_ber